In [7]:
from __future__ import annotations

import argparse
import os
from collections import Counter
from typing import Iterable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA


AA_ORDER = list("ACDEFGHIKLMNPQRSTVWY")
CLASS_LABEL_TO_NAME = {
    0: "Background",
    1: "S1A trypsin/chymotrypsin",
}


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--input",
        type=str,
        default="s1a_vs_background_dataset.csv",
    )
    parser.add_argument(
        "--outdir",
        type=str,
        default="qc_plots",
    )

    # 🔥 THIS IS THE FIX
    args, _ = parser.parse_known_args()
    return args


def ensure_outdir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def read_dataset(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)

    required_cols = {
        "sequence",
        "length",
        "class_label",
        "class_name",
        "organism",
        "has_ps00134",
        "has_ps00135",
        "has_both_catalytic_motifs",
    }
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Dataset is missing required columns: {sorted(missing)}")

    df["class_label"] = pd.to_numeric(df["class_label"], errors="coerce").astype(int)
    df["length"] = pd.to_numeric(df["length"], errors="coerce")

    for col in ["has_ps00134", "has_ps00135", "has_both_catalytic_motifs"]:
        df[col] = df[col].astype(bool)

    df["organism"] = df["organism"].fillna("Unknown organism")

    return df


def save_fig(path: str, dpi: int = 220) -> None:
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close()


def plot_length_distribution(df: pd.DataFrame, outdir: str) -> None:
    plt.figure(figsize=(9, 6))

    bg = df[df["class_label"] == 0]["length"].dropna()
    pos = df[df["class_label"] == 1]["length"].dropna()

    bins = np.arange(int(df["length"].min()), int(df["length"].max()) + 11, 10)

    plt.hist(
        bg,
        bins=bins,
        alpha=0.6,
        label=f"Background (n={len(bg):,})",
    )
    plt.hist(
        pos,
        bins=bins,
        alpha=0.6,
        label=f"S1A trypsin/chymotrypsin (n={len(pos):,})",
    )

    plt.xlabel("Sequence length (amino acids)")
    plt.ylabel("Number of sequences")
    plt.title("Sequence length distributions for S1A and background datasets")
    plt.legend()
    save_fig(os.path.join(outdir, "01_length_distribution.png"))


def build_positive_motif_classes(pos: pd.DataFrame) -> pd.Series:
    conditions = [
        pos["has_ps00134"] & pos["has_ps00135"],
        pos["has_ps00134"] & ~pos["has_ps00135"],
        ~pos["has_ps00134"] & pos["has_ps00135"],
        ~pos["has_ps00134"] & ~pos["has_ps00135"],
    ]
    choices = [
        "Both catalytic motifs\n(PS00134 + PS00135)",
        "Histidine motif only\n(PS00134 only)",
        "Serine motif only\n(PS00135 only)",
        "Neither catalytic motif",
    ]
    return pd.Series(np.select(conditions, choices, default="Unknown"), index=pos.index)


def plot_motif_breakdown(df: pd.DataFrame, outdir: str) -> None:
    pos = df[df["class_label"] == 1].copy()
    pos["motif_class"] = build_positive_motif_classes(pos)

    order = [
        "Both catalytic motifs\n(PS00134 + PS00135)",
        "Histidine motif only\n(PS00134 only)",
        "Serine motif only\n(PS00135 only)",
        "Neither catalytic motif",
    ]
    counts = pos["motif_class"].value_counts().reindex(order, fill_value=0)

    plt.figure(figsize=(9, 6))
    bars = plt.bar(range(len(counts)), counts.values)

    plt.xticks(range(len(counts)), counts.index)
    plt.ylabel("Number of S1A sequences")
    plt.xlabel("Catalytic motif class")
    plt.title("Catalytic motif composition within the S1A positive set")

    for i, v in enumerate(counts.values):
        plt.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=9)

    save_fig(os.path.join(outdir, "02_positive_motif_breakdown.png"))


def simplify_organism_name(name: str) -> str:
    """
    Keep organism labels readable in top-taxa bar plots.
    """
    name = str(name).strip()
    return name if len(name) <= 40 else name[:37] + "..."


def top_taxa_table(df: pd.DataFrame, top_n: int = 15) -> pd.DataFrame:
    counts = (
        df.groupby(["class_label", "organism"])
        .size()
        .reset_index(name="count")
    )

    rows = []
    for class_label in sorted(df["class_label"].unique()):
        sub = counts[counts["class_label"] == class_label].sort_values("count", ascending=False).head(top_n).copy()
        sub["organism"] = sub["organism"].map(simplify_organism_name)
        rows.append(sub)

    return pd.concat(rows, ignore_index=True)


def plot_top_taxa(df: pd.DataFrame, outdir: str, top_n: int = 15) -> None:
    taxa = top_taxa_table(df, top_n=top_n)

    fig, axes = plt.subplots(1, 2, figsize=(16, 8), sharex=False)

    for ax, class_label in zip(axes, [1, 0]):
        sub = taxa[taxa["class_label"] == class_label].sort_values("count", ascending=True)
        ax.barh(sub["organism"], sub["count"])
        ax.set_title(f"Top {top_n} taxa: {CLASS_LABEL_TO_NAME[class_label]}")
        ax.set_xlabel("Number of sequences")
        ax.set_ylabel("Organism")

    fig.suptitle("Most frequent organism labels in each class", y=1.02, fontsize=14)
    save_fig(os.path.join(outdir, "03_top_taxa_by_class.png"))


def aa_frequency_vector(seq: str) -> np.ndarray:
    seq = str(seq)
    counts = Counter(seq)
    length = len(seq)
    if length == 0:
        return np.zeros(len(AA_ORDER), dtype=float)
    return np.array([counts.get(aa, 0) / length for aa in AA_ORDER], dtype=float)


def compute_aa_matrix(df: pd.DataFrame) -> np.ndarray:
    return np.vstack(df["sequence"].map(aa_frequency_vector).values)


def plot_aa_composition_difference(df: pd.DataFrame, outdir: str) -> None:
    X = compute_aa_matrix(df)
    y = df["class_label"].values

    mean_pos = X[y == 1].mean(axis=0)
    mean_bg = X[y == 0].mean(axis=0)
    diff = mean_pos - mean_bg

    plt.figure(figsize=(11, 6))
    bars = plt.bar(range(len(AA_ORDER)), diff)

    plt.xticks(range(len(AA_ORDER)), AA_ORDER)
    plt.xlabel("Amino acid")
    plt.ylabel("Mean frequency difference (S1A - background)")
    plt.title("Difference in amino acid composition between S1A and background sequences")
    plt.axhline(0.0, linewidth=1)

    save_fig(os.path.join(outdir, "04_aa_composition_difference.png"))


def plot_pca_of_aa_composition(df: pd.DataFrame, outdir: str) -> None:
    X = compute_aa_matrix(df)
    y = df["class_label"].values

    pca = PCA(n_components=2)
    Z = pca.fit_transform(X)

    var1 = pca.explained_variance_ratio_[0] * 100
    var2 = pca.explained_variance_ratio_[1] * 100

    plt.figure(figsize=(9, 7))

    mask_bg = y == 0
    mask_pos = y == 1

    plt.scatter(
        Z[mask_bg, 0],
        Z[mask_bg, 1],
        alpha=0.25,
        s=10,
        label=f"Background (n={mask_bg.sum():,})",
    )
    plt.scatter(
        Z[mask_pos, 0],
        Z[mask_pos, 1],
        alpha=0.25,
        s=10,
        label=f"S1A trypsin/chymotrypsin (n={mask_pos.sum():,})",
    )

    plt.xlabel(f"PC1 ({var1:.1f}% variance explained)")
    plt.ylabel(f"PC2 ({var2:.1f}% variance explained)")
    plt.title("PCA of amino acid composition vectors")
    plt.legend(markerscale=2)
    save_fig(os.path.join(outdir, "05_pca_aa_composition.png"))


def print_summary(df: pd.DataFrame) -> None:
    print("\n=== Dataset summary ===")
    print(df["class_name"].value_counts(dropna=False))
    print("\n=== Motif summary in positives ===")
    pos = df[df["class_label"] == 1].copy()
    pos["motif_class"] = build_positive_motif_classes(pos)
    print(pos["motif_class"].value_counts())
    print("\n=== Length summary by class ===")
    print(df.groupby("class_label")["length"].describe())


def main() -> None:
    args = parse_args()
    ensure_outdir(args.outdir)

    df = read_dataset(args.input)
    print_summary(df)

    plot_length_distribution(df, args.outdir)
    plot_motif_breakdown(df, args.outdir)
    plot_top_taxa(df, args.outdir, top_n=15)
    plot_aa_composition_difference(df, args.outdir)
    plot_pca_of_aa_composition(df, args.outdir)

    print(f"\nWrote QC plots to: {args.outdir}")


if __name__ == "__main__":
    main()


=== Dataset summary ===
class_name
background                  35297
S1A_trypsin_chymotrypsin    19338
Name: count, dtype: int64

=== Motif summary in positives ===
motif_class
Both catalytic motifs\n(PS00134 + PS00135)    13959
Histidine motif only\n(PS00134 only)           2878
Neither catalytic motif                        1256
Serine motif only\n(PS00135 only)              1245
Name: count, dtype: int64

=== Length summary by class ===
               count        mean        std    min    25%    50%    75%    max
class_label                                                                   
0            35297.0  265.614811  34.301039  180.0  241.0  267.0  295.0  320.0
1            19338.0  261.173855  23.567808  180.0  247.0  259.0  273.0  320.0

Wrote QC plots to: qc_plots
